# Time series 11: LS-SVM / SVR (Support Vector Machine for spectrum prediction)

Uses **Support Vector Regression (SVR)** — and the closely related **Least Squares SVM (LS-SVM)** formulation — for multi-channel time series prediction. Each channel is predicted from its own lagged history (or optionally from all channels).

- **Features**: Past `SEQ_LEN` time steps per channel (univariate per channel) or flattened multi-channel window.
- **Model**: `sklearn.svm.SVR` with RBF kernel (C, gamma). LS-SVM uses equality constraints and a different solver; SVR is the standard SVM for regression and gives comparable use cases.
- **Split**: 60% train, 10% validation, 30% test (same as notebooks 8–10).
- **Evaluation**: MAPE, RMSE, MAE; comparison to naive (last value repeated).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
N = len(freq_bands)

Z = df[freq_bands].values.astype(np.float64)
Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
T_total = Z.shape[0]

print(f"Loaded {len(df)} rows. Channels (series): {N}")
print(f"Matrix shape: (T={T_total}, N={N})")

## Train / validation / test split (6 : 1 : 3)

Same split as time_series_8–10: 60% train, 10% validation, 30% test.

In [ ]:
train_ratio, val_ratio, test_ratio = 0.6, 0.1, 0.3
n_train = int(T_total * train_ratio)
n_val = int(T_total * val_ratio)
n_test = T_total - n_train - n_val

Z_train = Z[:n_train]
Z_val = Z[n_train : n_train + n_val]
Z_test = Z[n_train + n_val :]

test_start = n_train + n_val
print(f"Train: {n_train}, Val: {n_val}, Test: {n_test}")

## Lagged features and SVR per channel

For each channel, build sequences: input = last `SEQ_LEN` values, target = next value. Train one SVR per channel (univariate). Scale features with `StandardScaler` for better SVR performance.

In [2]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# Run the data-loading cell (cell 1) and the split cell (cell 3) first.
try:
    _ = N, Z, n_train
except NameError:
    raise RuntimeError("Run the data-loading cell (cell 1) and the train/val/test split cell (cell 3) first.")

SEQ_LEN = 24  # past hours to predict next hour
C = 1.0
GAMMA = "scale"  # 1 / (n_features * X.var())

# RBF SVR is O(n²)–O(n³) in training size. Cap samples so the notebook runs in minutes, not hours.
# Set to None to use all training data (will be very slow for 70 channels).
MAX_TRAIN_SAMPLES = 2000

def build_lagged(Z_mat, j, seq_len, start, end):
    """Channel j: X (n, seq_len), y (n,) from Z_mat[start:end]."""
    X_list, y_list = [], []
    for i in range(start + seq_len, end):
        X_list.append(Z_mat[i - seq_len : i, j])
        y_list.append(Z_mat[i, j])
    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float64), np.array(y_list, dtype=np.float64)

scalers = {}
models_svr = {}
for j in tqdm(range(N), desc="Training SVR per channel"):
    X_tr, y_tr = build_lagged(Z, j, SEQ_LEN, 0, n_train)
    if X_tr is None or len(X_tr) < 10:
        continue
    if MAX_TRAIN_SAMPLES is not None and len(X_tr) > MAX_TRAIN_SAMPLES:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X_tr), size=MAX_TRAIN_SAMPLES, replace=False)
        idx.sort()
        X_tr, y_tr = X_tr[idx], y_tr[idx]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    svr = SVR(kernel="rbf", C=C, gamma=GAMMA)
    svr.fit(X_tr_s, y_tr)
    scalers[j] = scaler
    models_svr[j] = svr

print(f"Trained {len(models_svr)} channel SVRs (max {MAX_TRAIN_SAMPLES or 'all'} train samples per channel).")

RuntimeError: Run the data-loading cell (cell 1) and the train/val/test split cell (cell 3) first.

## Test: rolling prediction and reassembly

For each test step, form the lagged input from the full history (train+val+test up to that point), scale, predict per channel, reassemble (n_test, N).

In [ ]:
pred_test = np.full((n_test, N), np.nan, dtype=np.float64)

for t in range(n_test):
    global_idx = test_start + t
    start = global_idx - SEQ_LEN
    if start < 0:
        continue
    for j in models_svr:
        x = Z[start:global_idx, j].reshape(1, -1)
        x_s = scalers[j].transform(x)
        pred_test[t, j] = models_svr[j].predict(x_s)[0]

# Fill channels without a model with persistence
for t in range(n_test):
    for j in range(N):
        if np.isnan(pred_test[t, j]):
            pred_test[t, j] = Z[test_start + t - 1, j] if t > 0 else Z[test_start - 1, j]

valid = ~np.isnan(pred_test)
pred_flat = pred_test[valid]
true_flat = Z_test[valid]

mape = np.mean(np.abs((true_flat - pred_flat) / (np.abs(true_flat) + 1e-8))) * 100
rmse = np.sqrt(np.mean((true_flat - pred_flat) ** 2))
mae = np.mean(np.abs(true_flat - pred_flat))

print("SVR (test set):")
print(f"  MAPE = {mape:.4f}%")
print(f"  RMSE = {rmse:.4f}")
print(f"  MAE  = {mae:.4f}")

# Naive: last value repeated
pred_naive = np.zeros((n_test, N), dtype=np.float64)
for t in range(n_test):
    pred_naive[t] = Z[test_start + t - 1] if t > 0 else Z[test_start - 1]
mape_naive = np.mean(np.abs((Z_test - pred_naive) / (np.abs(Z_test) + 1e-8))) * 100
rmse_naive = np.sqrt(np.mean((Z_test - pred_naive) ** 2))
mae_naive = np.mean(np.abs(Z_test - pred_naive))
print("Naive (last value repeated):")
print(f"  MAPE = {mape_naive:.4f}%, RMSE = {rmse_naive:.4f}, MAE = {mae_naive:.4f}")

## Comparison and MAE / MASE per step

Same style as time_series_8–10: summary comparison and plots of MAE and MASE per test step.

In [ ]:
print("Comparison: SVR vs Naive")
print(f"  MAPE: SVR = {mape:.4f}%,  Naive = {mape_naive:.4f}%")
print(f"  RMSE: SVR = {rmse:.4f},  Naive = {rmse_naive:.4f}")
print(f"  MAE:  SVR = {mae:.4f},  Naive = {mae_naive:.4f}")

import matplotlib.pyplot as plt
maes_svr = [float(np.mean(np.abs(Z_test[t] - pred_test[t]))) for t in range(n_test)]
maes_naive_d = [float(np.mean(np.abs(Z_test[t] - pred_naive[t]))) for t in range(n_test)]
mase_svr = [maes_svr[t] / (maes_naive_d[t] + 1e-12) for t in range(n_test)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
steps = range(1, n_test + 1)
ax1.plot(steps, maes_svr, "o-", label="SVR", markersize=2)
ax1.plot(steps, maes_naive_d, "^-", label="Naive", markersize=2)
ax1.set_xlabel("Test step")
ax1.set_ylabel("MAE")
ax1.set_title("MAE per step (test set)")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(steps, mase_svr, "o-", label="SVR", markersize=2)
ax2.axhline(1.0, color="gray", linestyle="--", label="Naive (MASE=1)")
ax2.set_xlabel("Test step")
ax2.set_ylabel("MASE")
ax2.set_title("MASE per step (test set)")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Save model (optional)

Save fitted SVRs and scalers to `data/models/lsvm_model.joblib` so you can skip retraining.

In [ ]:
import joblib

MODEL_DIR = _root / "data" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
save_path = MODEL_DIR / "lsvm_model.joblib"
joblib.dump({"models": models_svr, "scalers": scalers, "N": N, "SEQ_LEN": SEQ_LEN}, save_path)
print(f"Model saved to {save_path}")

## Load saved model (optional)

Run this cell instead of the training and test cells to load from `data/models/lsvm_model.joblib`, then run the comparison/plot cell.

In [ ]:
import joblib

load_path = _root / "data" / "models" / "lsvm_model.joblib"
if not load_path.exists():
    raise FileNotFoundError(f"No saved model at {load_path}. Run training and save first.")
ckpt = joblib.load(load_path)
models_svr = ckpt["models"]
scalers = ckpt["scalers"]
N_ld = ckpt["N"]
SEQ_LEN_ld = ckpt["SEQ_LEN"]
print(f"Loaded model from {load_path}. Run test and comparison cells next.")